# Sinh voice SC-001–SC-040 trên Kaggle: VoxCPM2 + ViMD

Dùng [`openbmb/VoxCPM2`](https://huggingface.co/openbmb/VoxCPM2) để sinh
`calls.<call_id>.customer_turns`. Chọn **một giọng ViMD đúng tỉnh/giới tính cho mỗi scenario**,
giữ giọng qua các call và xuất **một WAV cho mỗi phần tử**.
40 JSON hiện tại có **80 call / 250 lượt**. Ví dụ:
`audio/SC-001/call_1/SC-001_call_1_turn_001.wav`.
Manifest giữ index gốc 0-based, thứ tự 1-based, câu gốc/câu đọc, speaker, seed và trạng thái.

## Chạy trên Kaggle

1. Tạo một Kaggle Dataset từ **40 file `SC-001.json`–`SC-040.json`** trong
   `scenarios_complete dataset`, hoặc ZIP chứa 40 file này; **Add Input** vào notebook.
2. Import file `.ipynb` này vào Kaggle Notebook Editor.
3. Bật **Internet** và **Accelerator → GPU** trong **Settings / Session options**.
   Notebook dùng một GPU `cuda:0`, kể cả phiên có hai GPU. Cần Torch/Torchaudio ≥ 2.5, CUDA ≥ 12.
4. Giữ **`RUN_MODE='smoke'`**, chạy các ô từ trên xuống: chọn SC-001, lấy reference,
   sinh **3 WAV đầu** để nghe thử. Nếu ô cài đặt yêu cầu restart do package đã import,
   restart kernel/session rồi chạy lại từ đầu.
5. Sau khi nghe, đổi **`RUN_MODE='full'`**, chạy lại **từ phần 2 trở xuống** trong cùng session
   để chọn 40 giọng và sinh đủ 250 WAV. WAV đã hoàn thành được dùng lại khi fingerprint/checksum khớp.
6. Để lưu phiên đầy đủ, đặt `RUN_MODE='full'`, chọn **Save Version → Save & Run All**,
   chờ thành công rồi tải **`voxcpm2_vimd_customer_turns.zip`** trong tab Output.
   Phiên này dùng môi trường mới; attach output phiên trước và đặt `RESUME_FROM_DIR` nếu cần resume.

`DATASET_DIR=None` tự tìm thư mục đủ 40 JSON trong `/kaggle/input`, cũng hỗ trợ ZIP chưa giải nén.
Nếu có nhiều bản, đặt `DATASET_DIR` rõ ràng.
WAV/reference/manifest: `/kaggle/working/voxcpm2_vimd_SC001_SC040`.
Checkpoint/cache Hugging Face: `/tmp/voxcpm2_vimd_runtime`, không đóng vào ZIP output.
Lưu output version trước khi kết thúc session; working của phiên tương tác chưa phải output đã lưu.

**Mẫu audio mặc định ≥ 15 giây** (`REF_MIN_SECONDS=15.0`), tối đa 20 giây, ưu tiên gần 17 giây.
Mẫu mono được resample về **16 kHz**; VoxCPM2 xuất **48 kHz**.
Dùng chế độ **ref + continuation (Hi-Fi/Ultimate Cloning)**: cùng WAV làm
`reference_wav_path` và `prompt_wav_path`, kèm transcript đúng bản ghi ViMD.
Tắt VAD/denoise, giữ nguyên toàn bộ lời và khoảng lặng; AudioVAE chỉ pad để căn patch.
Không nạp ASR. Cache đặc trưng giọng một lần trong session để dùng lại cho các lượt.

`MODEL_DTYPE='auto'` dùng BF16 khi GPU hỗ trợ native BF16; GPU T4/P100 dùng FP32.
FP32 cần nhiều VRAM hơn. Notebook tắt `torch.compile` mặc định.
Nếu nạp model hết VRAM, chọn GPU hỗ trợ BF16 khi tài khoản có; có thể thử `MODEL_DTYPE='float16'`
trên T4 và nghe kỹ các WAV smoke trước khi chạy đủ. Chưa xác nhận suy luận GPU thực tế trong notebook này.

Giữ ánh xạ 40 tỉnh/giới tính như bản trước. SC-028 “Hà Tỉnh” được chuẩn hóa thành **Hà Tĩnh**;
nhãn vùng yêu cầu và vùng ViMD được ghi riêng. Nghe kiểm tra accent và cách đọc sau khi sinh.
Lần đầu lấy đủ 40 reference từ Parquet ViMD có thể truyền nhiều GB; cache reference/metadata
trong output giúp giảm tải lại ở các lần chạy sau.

Nguồn: [VoxCPM2 model card](https://huggingface.co/openbmb/VoxCPM2),
[API và chế độ cloning](https://voxcpm.readthedocs.io/en/latest/usage_guide.html),
[mã VoxCPM](https://github.com/OpenBMB/VoxCPM),
[ViMD](https://huggingface.co/datasets/nguyendv02/ViMD_Dataset).
Manifest ghi giấy phép nguồn: ViMD **CC BY-NC-ND 4.0**; mã/checkpoint VoxCPM2 **Apache 2.0**.


## 1. Kiểm tra GPU và cài dependency

Bật Internet/GPU. Dùng `voxcpm==2.0.3`, `transformers==4.57.3`; giữ bộ Torch/Torchaudio/CUDA
của Kaggle nếu đáp ứng ≥ 2.5 / CUDA ≥ 12. Kiểm tra import trong process riêng, chưa tải checkpoint.

Metadata của [voxcpm 2.0.3 trên PyPI](https://pypi.org/project/voxcpm/2.0.3/) yêu cầu Python ≥ 3.10,
không đặt giới hạn trên. README vẫn hướng dẫn Python 3.10–3.12; tương thích đầy đủ với Python 3.13
chưa được xác nhận ở đây. Notebook cho phép môi trường Kaggle Python 3.13.15 chạy bước cài đặt,
rồi kiểm tra dependency bằng pip và import thực tế. Nếu thất bại, log giữ nguyên lỗi gốc và phiên bản Python.

Nếu package đã được import rồi đổi phiên bản, restart kernel/session và chạy lại từ đầu.


In [ ]:
import importlib
import importlib.metadata
import os
from pathlib import Path
import shutil
import subprocess
import sys

if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này dùng cho Kaggle. Import vào Kaggle Notebook và attach 40 JSON bằng Add Input.')
print('Python:', sys.version)
print('Interpreter:', sys.executable)
if sys.version_info[:2] < (3, 10):
    raise RuntimeError(f'VoxCPM2 cần Python >= 3.10; hiện tại là {sys.version.split()[0]}.')

VOXCPM_PACKAGE_VERSION = '2.0.3'
TRANSFORMERS_VERSION = '4.57.3'
RUNTIME_DIR = Path('/tmp/voxcpm2_vimd_runtime')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(RUNTIME_DIR / 'huggingface')
os.environ['HF_HUB_CACHE'] = str(RUNTIME_DIR / 'huggingface' / 'hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = os.environ['HF_HUB_CACHE']

gpu_probe = subprocess.run(
    [sys.executable, '-c',
     'import torch, torchaudio; from packaging.version import Version; '
     'print("Torch:", torch.__version__, "Torchaudio:", torchaudio.__version__); '
     'assert Version(torch.__version__) >= Version("2.5"), "Torch >= 2.5 required"; '
     'assert Version(torchaudio.__version__) >= Version("2.5"), "Torchaudio >= 2.5 required"; '
     'assert torch.cuda.is_available(), "CUDA GPU is not available"; '
     'assert Version(torch.version.cuda) >= Version("12.0"), "CUDA >= 12.0 required"; '
     'print("GPU:", torch.cuda.get_device_name(0))'],
    capture_output=True, text=True,
)
print(gpu_probe.stdout)
if gpu_probe.returncode:
    print(gpu_probe.stderr)
    raise RuntimeError('Chưa có bộ Torch/Torchaudio CUDA hợp lệ. Bật GPU trong Kaggle rồi khởi động session mới.')

# Giữ đúng các wheel CUDA đi kèm môi trường, tránh pip thay bộ Torch trong lúc cài VoxCPM2.
constraints = []
for distribution in ['torch', 'torchaudio', 'torchvision']:
    try:
        constraints.append(distribution + '==' + importlib.metadata.version(distribution))
    except importlib.metadata.PackageNotFoundError:
        pass
constraint_file = RUNTIME_DIR / 'torch_constraints.txt'
constraint_file.write_text('\n'.join(constraints) + '\n', encoding='utf-8')

tracked_packages = ['torch', 'torchaudio', 'transformers', 'voxcpm', 'numpy',
                    'pandas', 'pyarrow', 'soundfile', 'librosa', 'huggingface_hub']
loaded_versions = {}
for name in tracked_packages:
    if name in sys.modules:
        try:
            loaded_versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            loaded_versions[name] = None

try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet', '--constraint', str(constraint_file),
        f'voxcpm=={VOXCPM_PACKAGE_VERSION}', f'transformers=={TRANSFORMERS_VERSION}',
        'pandas', 'pyarrow', 'fsspec', 'huggingface_hub', 'soundfile', 'librosa', 'tqdm',
    ])
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f'Cài dependency thất bại trên Python {sys.version.split()[0]}. '
        'Xem lỗi pip phía trên để xác định package hoặc xung đột phiên bản.'
    ) from exc
importlib.invalidate_caches()
changed = [name for name, version in loaded_versions.items()
           if importlib.metadata.version(name) != version]
if changed:
    raise RuntimeError(f'Package đã import vừa đổi phiên bản: {changed}. Restart kernel/session rồi chạy lại từ đầu.')

if shutil.which('ffmpeg') is None:
    subprocess.check_call(['apt-get', 'update', '-qq'])
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])

import_probe = subprocess.run(
    [sys.executable, '-c',
     'import inspect, torch, torchaudio, numpy, pandas, pyarrow, soundfile, librosa, transformers; '
     'from voxcpm import VoxCPM; from voxcpm.model.voxcpm2 import VoxCPM2Model; '
     'assert "trim_silence_vad" in inspect.signature(VoxCPM2Model.build_prompt_cache).parameters; '
     'assert "reference_wav_path" in inspect.signature(VoxCPM2Model.build_prompt_cache).parameters; '
     'assert hasattr(VoxCPM2Model, "generate_with_prompt_cache"); '
     'print("VoxCPM2 import OK; Transformers:", transformers.__version__); '
     'print("CUDA:", torch.cuda.is_available())'],
    capture_output=True, text=True,
)
print(import_probe.stdout)
if import_probe.returncode:
    print(import_probe.stderr)
    raise RuntimeError(
        f'Dependency chưa import được trên Python {sys.version.split()[0]}. '
        'Xem traceback phía trên để xác định dependency gây lỗi.'
    )


## 2. Cấu hình, input/output và resume

`RUN_MODE='smoke'`: SC-001 / 3 lượt đầu. `RUN_MODE='full'`: 40 scenario / mọi call / mọi lượt và ZIP.
Có thể chỉnh `SCENARIO_IDS`, `CALL_IDS`, `MAX_TURNS` nếu chỉ cần một nhóm.
`DATASET_DIR=None` tự tìm thư mục đủ 40 JSON hoặc ZIP; đặt rõ đường dẫn nếu có nhiều bản.
`/kaggle/input` chỉ để đọc; output phải nằm trong `/kaggle/working`.

`OVERWRITE=False` dùng lại WAV khi câu đọc/reference/cấu hình/checksum khớp.
Trong session mới, attach output đã lưu bằng Add Input và đặt `RESUME_FROM_DIR` là thư mục
chứa `manifest.json`, `audio`, `references`, `cache`, ví dụ
`/kaggle/input/<output-da-attach>/voxcpm2_vimd_SC001_SC040` (dùng đường dẫn thực tế).
Trong cùng session, để `RESUME_FROM_DIR=None`.

`REF_MIN_SECONDS=15.0`: chỉ nhận mẫu gốc **≥ 15 giây**; max 20 giây, target 17 giây.
Ngưỡng dành cho **audio mẫu**, không phải độ dài WAV cần sinh. Đổi đồng thời min/max/target
nếu đổi khoảng thời lượng rồi chạy lại từ phần 2. Mẫu được resample 16 kHz; file đầu ra là 48 kHz.
Mặc định lấy ViMD split `train`, không tự đổi tỉnh/giới tính khi thiếu mẫu.

`CFG_VALUE=2.0`, `INFERENCE_TIMESTEPS=10`. Model tự nhận ngôn ngữ từ câu tiếng Việt,
không truyền tham số language/speed. Chuẩn hóa số/ngày/tiền dùng hàm tiếng Việt ở phần 4,
không dùng external text normalizer của VoxCPM. Transcript reference giữ nguyên câu ViMD.

`MODEL_DTYPE='auto'`: BF16 nếu GPU có native BF16, FP32 trên T4/P100.
Có thể chủ động đặt `bfloat16`, `float32` hoặc `float16`; BF16 cần GPU hỗ trợ.
`OPTIMIZE_MODEL=False` tắt compile/warm-up. `MAX_GEN_TOKENS=1024` giới hạn số audio patch cần sinh,
không tính bằng giây. Nếu generation chạm giới hạn, lượt đó được đánh lỗi để không xuất nhầm câu bị cắt.
Notebook tự retry bad case tối đa 3 lần theo cơ chế của model.


In [ ]:
import hashlib
import shutil
import zipfile

RUN_MODE = 'smoke'  # Đổi thành 'full' sau khi nghe 3 WAV thử.
DATASET_DIR = None  # Hoặc Path('/kaggle/input/<dataset>/<thu-muc-chua-JSON>')
OUTPUT_DIR = Path('/kaggle/working/voxcpm2_vimd_SC001_SC040')
MODEL_CACHE_DIR = RUNTIME_DIR / 'model'
RESUME_FROM_DIR = None  # Thư mục output cũ đã attach vào /kaggle/input, nếu có.

if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError("RUN_MODE phải là 'smoke' hoặc 'full'.")
SCENARIO_IDS = ['SC-001'] if RUN_MODE == 'smoke' else None
CALL_IDS = None
MAX_TURNS = 3 if RUN_MODE == 'smoke' else None
RUN_TTS = True
OVERWRITE = False

def resolve_dataset_dir(configured_dir, input_root, extraction_root):
    required_names = {f'SC-{i:03d}.json' for i in range(1, 41)}
    if configured_dir is not None:
        configured_dir = Path(configured_dir)
        if not configured_dir.is_dir():
            raise FileNotFoundError(f'DATASET_DIR không tồn tại: {configured_dir}')
        missing = sorted(name for name in required_names if not (configured_dir / name).is_file())
        if missing:
            raise FileNotFoundError(f'DATASET_DIR thiếu JSON: {missing}. Thư mục phải chứa trực tiếp 40 JSON.')
        return configured_dir
    candidates = sorted({p.parent for p in input_root.rglob('SC-001.json')
                         if all((p.parent / name).is_file() for name in required_names)})
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(f'Tìm thấy nhiều thư mục scenario: {candidates}. Đặt DATASET_DIR rõ ràng.')
    # Kaggle có thể giữ ZIP hoặc giải nén input; hỗ trợ cả hai cách.
    archives = []
    for archive_path in sorted(input_root.rglob('*.zip')):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                members = {}
                for info in archive.infolist():
                    if not info.is_dir():
                        members.setdefault(Path(info.filename).name, []).append(info.filename)
                if all(len(members.get(name, [])) == 1 for name in required_names):
                    archives.append((archive_path, {name: members[name][0] for name in required_names}))
        except zipfile.BadZipFile:
            continue
    if len(archives) > 1:
        raise ValueError('Có nhiều ZIP chứa đủ 40 scenario. Chỉ attach một bộ hoặc đặt DATASET_DIR đã giải nén.')
    if len(archives) == 1:
        archive_path, members = archives[0]
        directory = extraction_root / ('scenarios_' + hashlib.sha256(str(archive_path).encode()).hexdigest()[:12])
        with zipfile.ZipFile(archive_path) as archive:
            sizes = [archive.getinfo(member).file_size for member in members.values()]
            if max(sizes) > 5 * 1024 ** 2 or sum(sizes) > 50 * 1024 ** 2:
                raise ValueError('40 JSON trong ZIP vượt giới hạn dung lượng của notebook.')
            directory.mkdir(parents=True, exist_ok=True)
            for name, member in members.items():
                # Ghi tên SC cố định; không extract đường dẫn do ZIP cung cấp.
                (directory / name).write_bytes(archive.read(member))
        print('Đã đọc 40 JSON từ ZIP:', archive_path)
        return directory
    raise FileNotFoundError('Chưa thấy đủ SC-001.json–SC-040.json trong /kaggle/input. '
                            'Dùng Add Input để attach dataset/ZIP, hoặc sửa DATASET_DIR.')

def restore_previous_output(source_dir, destination_dir):
    source_dir, destination_dir = Path(source_dir).resolve(), Path(destination_dir).resolve()
    if source_dir == destination_dir:
        return 0
    if not (source_dir / 'manifest.json').is_file():
        raise FileNotFoundError(f'RESUME_FROM_DIR không chứa manifest.json: {source_dir}')
    files = [source_dir / 'manifest.json']
    for folder in ['audio', 'references', 'cache']:
        files.extend(p for p in (source_dir / folder).rglob('*') if p.is_file())
    copied = 0
    for path in files:
        if path.is_symlink() or '.partial' in path.name:
            continue
        resolved = path.resolve()
        if not resolved.is_relative_to(source_dir):
            raise ValueError('File resume nằm ngoài thư mục nguồn.')
        destination = destination_dir / path.relative_to(source_dir)
        if not destination.resolve().is_relative_to(destination_dir):
            raise ValueError('File resume nằm ngoài thư mục output.')
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, destination)
            copied += 1
    return copied

VIMD_DATASET_ID = 'nguyendv02/ViMD_Dataset'
VIMD_REVISION = '3a5b30157034e7eadd5c75fae1a820c6f9383398'
REFERENCE_SPLITS = ['train']  # Có thể chủ động thêm 'valid', 'test'.
REFERENCE_FILENAME_OVERRIDES = {}
REF_MIN_SECONDS = 15.0  # Ngưỡng tối thiểu của audio mẫu: duration >= ngưỡng này.
REF_MAX_SECONDS = 20.0  # Mẫu nguyên bản tối đa 20 giây; không cắt lời.
REF_TARGET_SECONDS = 17.0  # Ưu tiên mẫu gần 17 giây trong khoảng hợp lệ.
MAX_REFERENCE_GROUPS = 4

REFERENCE_SAMPLE_RATE = 16000
MODEL_REPO_ID = 'openbmb/VoxCPM2'
MODEL_REVISION = '32279effe8c19989596f05d353d1447f51d9e915'
MODEL_DTYPE = 'auto'  # BF16 khi hỗ trợ native; FP32 trên T4/P100.
OPTIMIZE_MODEL = False
CFG_VALUE = 2.0
INFERENCE_TIMESTEPS = 10
MIN_GEN_TOKENS = 2
MAX_GEN_TOKENS = 1024
BASE_SEED = 20261007

GENERATION_OPTIONS = {
    'cfg_value': CFG_VALUE, 'inference_timesteps': INFERENCE_TIMESTEPS,
    'min_len': MIN_GEN_TOKENS, 'max_len': MAX_GEN_TOKENS,
    'retry_badcase': True, 'retry_badcase_max_times': 3,
    'retry_badcase_ratio_threshold': 6.0,
}

# Chỉ áp dụng cho câu cần sinh, không áp dụng cho transcript reference.
NORMALIZE_NUMBERS = True
PRONUNCIATION_REPLACEMENTS = {'COD': 'si ô đi'}  # Thêm tên sản phẩm nếu cần.
TEXT_OVERRIDES = {}  # Ví dụ {'SC-001/call_1/0': 'Câu đọc chỉnh thủ công ...'}.

PREVIEW_SCENARIO_IDS = ['SC-001', 'SC-002', 'SC-028']
EXPORT_ZIP = RUN_MODE == 'full'  # Chế độ full tự xuất ZIP khi đủ mọi lượt.


if not OUTPUT_DIR.resolve().is_relative_to(Path('/kaggle/working').resolve()):
    raise ValueError('OUTPUT_DIR phải nằm trong /kaggle/working, không ghi vào input.')
if CFG_VALUE <= 0 or not isinstance(INFERENCE_TIMESTEPS, int) or INFERENCE_TIMESTEPS <= 0:
    raise ValueError('Cần CFG_VALUE > 0 và INFERENCE_TIMESTEPS là số nguyên > 0.')
if MODEL_DTYPE not in {'auto', 'bfloat16', 'float16', 'float32'}:
    raise ValueError('MODEL_DTYPE phải là auto/bfloat16/float16/float32.')
if not all(isinstance(value, int) for value in [MIN_GEN_TOKENS, MAX_GEN_TOKENS]):
    raise ValueError('MIN_GEN_TOKENS/MAX_GEN_TOKENS phải là số nguyên.')
if not 0 < MIN_GEN_TOKENS < MAX_GEN_TOKENS:
    raise ValueError('Cần 0 < MIN_GEN_TOKENS < MAX_GEN_TOKENS.')
if GENERATION_OPTIONS['retry_badcase_max_times'] < 1 or GENERATION_OPTIONS['retry_badcase_ratio_threshold'] <= 0:
    raise ValueError('Cần retry_badcase_max_times >= 1 và retry_badcase_ratio_threshold > 0.')
if not 0 < REF_MIN_SECONDS <= REF_MAX_SECONDS:
    raise ValueError('Cần 0 < REF_MIN_SECONDS <= REF_MAX_SECONDS.')
if not REF_MIN_SECONDS <= REF_TARGET_SECONDS <= REF_MAX_SECONDS:
    raise ValueError('REF_TARGET_SECONDS phải nằm giữa REF_MIN_SECONDS và REF_MAX_SECONDS.')
if MAX_TURNS is not None and (not isinstance(MAX_TURNS, int) or MAX_TURNS <= 0):
    raise ValueError('MAX_TURNS phải là số nguyên > 0 hoặc None.')
if MAX_REFERENCE_GROUPS < 1:
    raise ValueError('MAX_REFERENCE_GROUPS phải >= 1.')
DATASET_DIR = resolve_dataset_dir(DATASET_DIR, Path('/kaggle/input'), RUNTIME_DIR / 'scenario_inputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_FROM_DIR is not None:
    print('Số file khôi phục:', restore_previous_output(RESUME_FROM_DIR, OUTPUT_DIR))
print('Chế độ:', RUN_MODE)
print('JSON:', DATASET_DIR)
print('Output:', OUTPUT_DIR)
print('Dự kiến:', 'SC-001 / 3 lượt thử' if RUN_MODE == 'smoke' else '40 scenario / mọi customer_turns')


## 3. Danh sách giọng theo yêu cầu

In [ ]:
import gc
import hashlib
import io
import json
import os
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from huggingface_hub import HfFileSystem, snapshot_download
from IPython.display import Audio, display
from tqdm.auto import tqdm

VOICE_ROWS = [
    ('SC-001', 'Nữ', 'Nghệ An', 'Miền trung'),
    ('SC-002', 'Nam', 'Cao Bằng', 'Miền bắc'),
    ('SC-003', 'Nữ', 'Quảng Bình', 'Miền trung'),
    ('SC-004', 'Nam', 'Đà Nẵng', 'Miền trung'),
    ('SC-005', 'Nữ', 'Quảng Ngãi', 'Miền trung'),
    ('SC-006', 'Nam', 'Thái Bình', 'Miền bắc'),
    ('SC-007', 'Nữ', 'Bình Thuận', 'Miền trung'),
    ('SC-008', 'Nam', 'Hà Giang', 'Miền bắc'),
    ('SC-009', 'Nữ', 'Bình Phước', 'Miền nam'),
    ('SC-010', 'Nam', 'Bà Rịa Vũng Tàu', 'Miền nam'),
    ('SC-011', 'Nữ', 'Hồ Chí Minh', 'Miền nam'),
    ('SC-012', 'Nam', 'Hà Nội', 'Miền bắc'),
    ('SC-013', 'Nữ', 'Quảng Ninh', 'Miền bắc'),
    ('SC-014', 'Nam', 'Bình Dương', 'Miền nam'),
    ('SC-015', 'Nữ', 'Gia Lai', 'Miền trung'),
    ('SC-016', 'Nam', 'Thái Nguyên', 'Miền bắc'),
    ('SC-017', 'Nữ', 'Ninh Thuận', 'Miền trung'),
    ('SC-018', 'Nam', 'Lào Cai', 'Miền bắc'),
    ('SC-019', 'Nữ', 'Phú Thọ', 'Miền bắc'),
    ('SC-020', 'Nam', 'Hậu Giang', 'Miền nam'),
    ('SC-021', 'Nữ', 'Long An', 'Miền nam'),
    ('SC-022', 'Nam', 'Sơn La', 'Miền bắc'),
    ('SC-023', 'Nữ', 'Bình Định', 'Miền trung'),
    ('SC-024', 'Nam', 'Đắk Lắk', 'Miền trung'),
    ('SC-025', 'Nữ', 'Hải Phòng', 'Miền bắc'),
    ('SC-026', 'Nam', 'Kon Tum', 'Miền trung'),
    ('SC-027', 'Nữ', 'Phú Yên', 'Miền trung'),
    ('SC-028', 'Nam', 'Hà Tĩnh', 'Miền bắc'),
    ('SC-029', 'Nữ', 'Đắk Nông', 'Miền trung'),
    ('SC-030', 'Nam', 'Lạng Sơn', 'Miền bắc'),
    ('SC-031', 'Nữ', 'Thừa Thiên Huế', 'Miền trung'),
    ('SC-032', 'Nam', 'Lâm Đồng', 'Miền trung'),
    ('SC-033', 'Nữ', 'Tây Ninh', 'Miền nam'),
    ('SC-034', 'Nam', 'Cà Mau', 'Miền nam'),
    ('SC-035', 'Nữ', 'Nam Định', 'Miền bắc'),
    ('SC-036', 'Nam', 'Thanh Hóa', 'Miền trung'),
    ('SC-037', 'Nữ', 'Lai Châu', 'Miền bắc'),
    ('SC-038', 'Nam', 'Trà Vinh', 'Miền nam'),
    ('SC-039', 'Nữ', 'Quảng Trị', 'Miền trung'),
    ('SC-040', 'Nam', 'Quảng Nam', 'Miền trung'),
]

def province_key(value):
    value = str(value).replace('Đ', 'D').replace('đ', 'd')
    value = unicodedata.normalize('NFKD', value)
    return re.sub('[^a-z]', '', ''.join(c for c in value if not unicodedata.combining(c)).lower())

VOICE_SPECS = {
    sid: {'scenario_id': sid, 'gender_label': gender, 'gender': 0 if gender == 'Nữ' else 1,
          'province_requested': province, 'province_key': province_key(province),
          'region_requested': region}
    for sid, gender, province, region in VOICE_ROWS
}
if SCENARIO_IDS is None:
    SELECTED_IDS = list(VOICE_SPECS)
else:
    if not isinstance(SCENARIO_IDS, list) or not SCENARIO_IDS:
        raise ValueError('SCENARIO_IDS phải là list không rỗng hoặc None.')
    SELECTED_IDS = list(dict.fromkeys(SCENARIO_IDS))
    missing = set(SELECTED_IDS) - set(VOICE_SPECS)
    if missing:
        raise ValueError(f'Chỉ cấu hình SC-001 đến SC-040; ID không hợp lệ: {missing}')
if CALL_IDS is not None and (not isinstance(CALL_IDS, list) or not CALL_IDS):
    raise ValueError('CALL_IDS phải là list không rỗng hoặc None.')
display(pd.DataFrame([VOICE_SPECS[sid] for sid in SELECTED_IDS]).drop(columns='province_key'))


## 4. Đọc JSON và xem trước từng lượt

Chỉ lấy `customer_turns`. Các trường `notes`, `facts_established`, `success_if`, ...
là dữ liệu kịch bản, không được đưa vào TTS hay thực hiện như chỉ thị.
Nếu có lượt rỗng/sai kiểu, dừng kiểm tra để tránh làm dịch index.

Mặc định chuẩn hóa số tiền, ngày ISO, ngày `dd/mm`, giờ `hh:mm`, số điện thoại và số nguyên
thành cách đọc tiếng Việt; câu gốc luôn được giữ trong manifest. `TEXT_OVERRIDES` dùng khóa
`scenario_id/call_id/turn_index` (index **0-based**) cho cách đọc riêng; đặt
`NORMALIZE_NUMBERS=False` và `PRONUNCIATION_REPLACEMENTS={}` nếu muốn đưa nguyên văn vào model.

In [ ]:
DIGITS = ['không', 'một', 'hai', 'ba', 'bốn', 'năm', 'sáu', 'bảy', 'tám', 'chín']

def read_triplet(number, padded=False):
    hundred, rest = divmod(number, 100)
    ten, unit = divmod(rest, 10)
    words = []
    if hundred or padded:
        words += [DIGITS[hundred], 'trăm']
    if ten >= 2:
        words += [DIGITS[ten], 'mươi']
    elif ten == 1:
        words += ['mười']
    elif unit and (hundred or padded):
        words += ['linh']
    if unit:
        words += ['mốt' if unit == 1 and ten >= 2 else 'lăm' if unit == 5 and ten >= 1 else DIGITS[unit]]
    return ' '.join(words)

def read_integer(number):
    number = int(number)
    if number == 0:
        return DIGITS[0]
    if number < 0:
        return 'âm ' + read_integer(-number)
    units = ['', 'nghìn', 'triệu', 'tỷ', 'nghìn tỷ', 'triệu tỷ']
    groups = []
    while number:
        number, remainder = divmod(number, 1000)
        groups.append(remainder)
    if len(groups) > len(units):
        raise ValueError('Số vượt giới hạn bộ đọc số; dùng TEXT_OVERRIDES.')
    result = []
    for i in reversed(range(len(groups))):
        if groups[i]:
            result.append(read_triplet(groups[i], padded=bool(result) and groups[i] < 100))
            if units[i]:
                result.append(units[i])
    return ' '.join(result)

def normalize_spoken_text(text):
    text = unicodedata.normalize('NFC', text).strip()
    if NORMALIZE_NUMBERS:
        # Thay ngày trước số để không mất cấu trúc ngày tháng.
        text = re.sub(r'\b(\d{4})-(\d{2})-(\d{2})\b',
                      lambda m: f'{read_integer(m[3])} tháng {read_integer(m[2])} năm {read_integer(m[1])}', text)
        text = re.sub(r'\b(\d{1,2})/(\d{1,2})(?:/(\d{4}))?\b',
                      lambda m: f'{read_integer(m[1])} tháng {read_integer(m[2])}' +
                      (f' năm {read_integer(m[3])}' if m[3] else ''), text)
        text = re.sub(r'\b(\d{1,2}):(\d{2})\b',
                      lambda m: f'{read_integer(m[1])} giờ' +
                      (f' {read_integer(m[2])} phút' if int(m[2]) else ''), text)
        text = re.sub(r'(?<!\d)\d{1,3}(?:\.\d{3})+(?!\d)',
                      lambda m: read_integer(m[0].replace('.', '')), text)
        text = re.sub(r'(?<!\w)0\d{8,10}(?!\w)',
                      lambda m: ' '.join(DIGITS[int(d)] for d in m[0]), text)
        text = re.sub(r'\d+,\d+', lambda m: read_integer(m[0].split(',')[0]) +
                      ' phẩy ' + ' '.join(DIGITS[int(d)] for d in m[0].split(',')[1]), text)
        text = re.sub(r'\d+', lambda m: read_integer(m[0]), text)
    for source, target in PRONUNCIATION_REPLACEMENTS.items():
        text = re.sub(r'(?<!\w)' + re.escape(source) + r'(?!\w)', target, text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).lower().strip()

def safe_component(value):
    # Không cho tên call trong JSON ghi ra ngoài thư mục output.
    value = str(value)
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', value):
        raise ValueError(f'ID không an toàn để đặt tên file: {value!r}')
    return value

def build_tasks(dataset_dir, selected_ids):
    tasks = []
    for sid in selected_ids:
        source_file = dataset_dir / f'{sid}.json'
        payload = json.loads(source_file.read_text(encoding='utf-8-sig'))
        if not isinstance(payload, dict) or payload.get('scenario_id', sid) != sid:
            raise ValueError(f'{source_file.name}: scenario_id không khớp tên file.')
        calls = payload.get('calls')
        if not isinstance(calls, dict) or not calls:
            raise ValueError(f'{sid}: thiếu calls hoặc calls rỗng.')
        chosen_calls = list(calls) if CALL_IDS is None else list(dict.fromkeys(CALL_IDS))
        for call_id in chosen_calls:
            safe_component(call_id)
            if call_id not in calls or not isinstance(calls[call_id], dict):
                raise ValueError(f'{sid}: call không hợp lệ: {call_id}')
            turns = calls[call_id].get('customer_turns')
            if not isinstance(turns, list) or not turns:
                raise ValueError(f'{sid}/{call_id}: customer_turns phải là list không rỗng.')
            for index, original in enumerate(turns):
                if not isinstance(original, str) or not original.strip():
                    raise ValueError(f'{sid}/{call_id}/index={index}: lượt rỗng/sai kiểu; không tự bỏ để giữ index.')
                task_id = f'{sid}/{call_id}/{index}'
                spoken = TEXT_OVERRIDES.get(task_id, normalize_spoken_text(original))
                if not isinstance(spoken, str) or not spoken.strip():
                    raise ValueError(f'TEXT_OVERRIDES rỗng/sai kiểu: {task_id}')
                relative = f'audio/{sid}/{call_id}/{sid}_{call_id}_turn_{index + 1:03d}.wav'
                tasks.append({'task_id': task_id, 'scenario_id': sid, 'call_id': call_id,
                              'turn_index': index, 'turn_number': index + 1,
                              'source_file': str(source_file), 'text_original': original,
                              'text_tts': spoken, 'output_relative': relative})
    return tasks

TASKS = build_tasks(DATASET_DIR, SELECTED_IDS)
summary = pd.DataFrame(TASKS).groupby('scenario_id').agg(
    calls=('call_id', 'nunique'), customer_turns=('task_id', 'count'))
display(summary)
print(f'Tổng: {len(SELECTED_IDS)} scenario / {summary.calls.sum()} call / {len(TASKS)} file WAV.')
display(pd.DataFrame(TASKS)[['task_id', 'text_original', 'text_tts', 'output_relative']].head(8))

## 5. Đọc metadata ViMD, không đọc cột audio

Metadata nằm trong `OUTPUT_DIR/cache`, gắn revision và split. Dùng các cột text/nhãn từ
Parquet gốc, đọc tối đa 4 shard đồng thời. Mỗi ứng viên lưu shard, row group và index
để lấy đúng audio; `Nghệ An` và `NgheAn` được khớp bằng tên đã chuẩn hóa.
Cache này được đưa vào ZIP output để dùng lại ở phiên khác. Lỗi mạng: kiểm tra Internet,
rồi chạy lại ô này. Khi cache khớp cấu hình, notebook không quét metadata lại.

In [ ]:
def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    for attempt in range(5):
        try:
            temporary.replace(path)
            break
        except PermissionError:
            if attempt == 4:
                raise
            time.sleep(0.1 * (attempt + 1))

def fingerprint(data):
    return hashlib.sha256(json.dumps(data, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

HF_FS = HfFileSystem(token=os.getenv('HF_TOKEN'))
METADATA_DIR = OUTPUT_DIR / 'cache'
METADATA_DIR.mkdir(parents=True, exist_ok=True)
METADATA_FILE = METADATA_DIR / 'vimd_metadata.parquet'
METADATA_CONFIG_FILE = METADATA_DIR / 'vimd_metadata_config.json'
metadata_config = {'version': 1, 'dataset_id': VIMD_DATASET_ID,
                   'revision': VIMD_REVISION, 'splits': REFERENCE_SPLITS}

def load_vimd_metadata():
    if METADATA_FILE.is_file() and METADATA_CONFIG_FILE.is_file():
        if json.loads(METADATA_CONFIG_FILE.read_text(encoding='utf-8')) == metadata_config:
            print('Dùng cache metadata:', METADATA_FILE)
            return pd.read_parquet(METADATA_FILE)
    if not REFERENCE_SPLITS or set(REFERENCE_SPLITS) - {'train', 'valid', 'test'}:
        raise ValueError('REFERENCE_SPLITS chỉ nhận train, valid, test và không được rỗng.')
    root = f'datasets/{VIMD_DATASET_ID}@{VIMD_REVISION}'
    paths = sorted(HF_FS.glob(f'{root}/data/*.parquet'))
    paths = [p for p in paths if p.rsplit('/', 1)[-1].split('-', 1)[0] in REFERENCE_SPLITS]
    if not paths:
        raise FileNotFoundError('Không thấy Parquet gốc ở revision/split đã chọn.')
    columns = ['region', 'province_code', 'province_name', 'filename', 'text', 'speakerID', 'gender']
    def read_metadata_shard(path):
        shard_frames = []
        with HF_FS.open(path, 'rb', block_size=65536, cache_type='readahead') as stream:
            parquet = pq.ParquetFile(stream)
            missing = set(columns + ['audio']) - set(parquet.schema_arrow.names)
            if missing:
                raise ValueError(f'Schema ViMD thiếu cột: {missing}')
            for group_index in range(parquet.num_row_groups):
                frame = parquet.read_row_group(group_index, columns=columns).to_pandas()
                frame['source_parquet'] = path
                frame['row_group'] = group_index
                frame['row_in_group'] = np.arange(len(frame))
                frame['split'] = path.rsplit('/', 1)[-1].split('-', 1)[0]
                shard_frames.append(frame)
        return shard_frames
    frames = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        for shard_frames in tqdm(pool.map(read_metadata_shard, paths), total=len(paths),
                                 desc='Metadata ViMD (không audio)'):
            frames.extend(shard_frames)
    metadata = pd.concat(frames, ignore_index=True)
    temp = METADATA_FILE.with_name(METADATA_FILE.name + '.partial')
    metadata.to_parquet(temp, index=False)
    temp.replace(METADATA_FILE)
    atomic_json(METADATA_CONFIG_FILE, metadata_config)
    return metadata

VIMD_METADATA = load_vimd_metadata()
VIMD_METADATA['province_key'] = VIMD_METADATA.province_name.map(province_key)
VIMD_METADATA['word_count'] = VIMD_METADATA.text.fillna('').str.split().str.len()
VIMD_METADATA['gender'] = pd.to_numeric(VIMD_METADATA.gender, errors='raise').astype(int)

coverage = []
for sid in SELECTED_IDS:
    spec = VOICE_SPECS[sid]
    matches = VIMD_METADATA[(VIMD_METADATA.province_key == spec['province_key']) &
                            (VIMD_METADATA.gender == spec['gender'])]
    coverage.append({'scenario_id': sid, 'province_requested': spec['province_requested'],
                     'gender_requested': spec['gender_label'], 'candidates': len(matches),
                     'province_vimd': ', '.join(sorted(matches.province_name.unique())),
                     'region_vimd': ', '.join(sorted(matches.region.unique()))})
COVERAGE = pd.DataFrame(coverage)
display(COVERAGE)
if (COVERAGE.candidates == 0).any():
    missing = COVERAGE.loc[COVERAGE.candidates == 0, 'scenario_id'].tolist()
    raise ValueError(f'Không có giọng đúng tỉnh/giới tính: {missing}. Kiểm tra tên tỉnh hoặc thêm split; không dùng giọng khác thay thế.')
print(f'Metadata: {len(VIMD_METADATA):,} bản ghi; đủ tỉnh/giới tính cho {len(SELECTED_IDS)} scenario.')

## 6. Chọn và lưu một reference cho mỗi scenario

Chọn mẫu nguyên bản đúng tỉnh/giới tính có thời lượng **≥ `REF_MIN_SECONDS` (15 giây)**,
không vượt `REF_MAX_SECONDS` (20 giây), ưu tiên gần target 17 giây.
Lọc im lặng/clipping nặng, đọc mono/resample về **16 kHz**. Transcript lấy đúng bản ghi ViMD.
Không cắt lời, không ghép mẫu, không nhận dạng lại transcript.
VoxCPM2 `build_prompt_cache(..., trim_silence_vad=False)` giữ toàn bộ waveform và chỉ pad
để căn patch AudioVAE. Cùng reference dùng cho cả ref và continuation.

Nghe để phát hiện nhạc nền, nhiều người nói hoặc transcript sai.
Mỗi lần chỉ giữ một audio row group trong RAM. Nếu thiếu mẫu hợp lệ, ghi lỗi và dừng trước generation.
Có thể tăng `MAX_REFERENCE_GROUPS` hoặc đặt `REFERENCE_FILENAME_OVERRIDES` rồi chạy lại.


In [ ]:
REFERENCE_DIR = OUTPUT_DIR / 'references'
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_BANK_FILE = REFERENCE_DIR / 'reference_bank.json'

class AudioGroupReader:
    def __init__(self, filesystem):
        self.filesystem = filesystem
        self.key = None
        self.table = None

    def get(self, row):
        key = (row['source_parquet'], int(row['row_group']))
        if key != self.key:
            self.table = None
            self.key = None
            gc.collect()
            with self.filesystem.open(key[0], 'rb', block_size=65536, cache_type='none') as stream:
                self.table = pq.ParquetFile(stream).read_row_group(key[1], columns=['audio'])
            self.key = key
        item = self.table.column('audio')[int(row['row_in_group'])].as_py()
        raw_bytes = item.get('bytes') if isinstance(item, dict) else None
        if not raw_bytes:
            raise ValueError('Audio không có embedded bytes; không tự dùng đường dẫn khác.')
        waveform, sample_rate = sf.read(io.BytesIO(raw_bytes), dtype='float32', always_2d=True)
        return waveform.mean(axis=1), int(sample_rate)

def audio_metrics(waveform, sample_rate):
    if not len(waveform) or sample_rate <= 0 or not np.isfinite(waveform).all():
        raise ValueError('Audio rỗng hoặc có NaN/Inf.')
    return {'duration_seconds': float(len(waveform) / sample_rate),
            'rms': float(np.sqrt(np.mean(waveform ** 2))),
            'peak': float(np.max(np.abs(waveform))),
            'clipping_fraction': float(np.mean(np.abs(waveform) >= 0.999))}

def reference_duration_is_valid(duration_seconds):
    return REF_MIN_SECONDS <= duration_seconds <= REF_MAX_SECONDS


def reference_request(spec):
    return {'dataset_id': VIMD_DATASET_ID, 'revision': VIMD_REVISION,
            'splits': REFERENCE_SPLITS, 'province_key': spec['province_key'],
            'gender': spec['gender'], 'filename_override': REFERENCE_FILENAME_OVERRIDES.get(spec['scenario_id']),
            'min_seconds': REF_MIN_SECONDS, 'max_seconds': REF_MAX_SECONDS,
            'target_seconds': REF_TARGET_SECONDS,
            'reference_sample_rate': REFERENCE_SAMPLE_RATE, 'selection_version': 3}

def choose_reference(spec, reader):
    sid = spec['scenario_id']
    pool = VIMD_METADATA[(VIMD_METADATA.province_key == spec['province_key']) &
                         (VIMD_METADATA.gender == spec['gender'])].copy()
    pool = pool[pool.text.notna() & pool.text.str.strip().ne('') & pool.speakerID.notna()]
    override = REFERENCE_FILENAME_OVERRIDES.get(sid)
    if override:
        pool = pool[pool.filename == override]
        if pool.empty:
            raise ValueError(f'{sid}: override {override} không đúng tỉnh/giới tính/split.')
    target_words = max(1, round(REF_TARGET_SECONDS * 3.5))  # Chỉ dùng để ưu tiên thứ tự tìm.
    pool['distance'] = (pool.word_count - target_words).abs()
    pool = pool.sort_values(['distance', 'filename', 'split'], kind='stable')
    groups = pool[['source_parquet', 'row_group']].drop_duplicates().head(MAX_REFERENCE_GROUPS)
    rejected = []
    for group in groups.to_dict('records'):
        group_pool = pool[(pool.source_parquet == group['source_parquet']) &
                          (pool.row_group == group['row_group'])]
        # Xét mọi transcript đúng nhãn trong group đã tải; lọc bằng thời lượng đo thực tế.
        valid = []
        for row in group_pool.to_dict('records'):
            waveform, sample_rate = reader.get(row)
            metrics = audio_metrics(waveform, sample_rate)
            if not reference_duration_is_valid(metrics['duration_seconds']):
                rejected.append(f"{row['filename']}: {metrics['duration_seconds']:.2f}s")
                continue
            if metrics['rms'] < 0.005 or metrics['clipping_fraction'] > 0.01:
                rejected.append(f"{row['filename']}: im lặng/clipping")
                continue
            score = abs(metrics['duration_seconds'] - REF_TARGET_SECONDS)
            valid.append((score, row, waveform, sample_rate, metrics))
        if not valid:
            continue
        _, row, waveform, source_sr, metrics = min(valid, key=lambda item: (item[0], item[1]['filename']))
        if source_sr != REFERENCE_SAMPLE_RATE:
            import librosa
            waveform = librosa.resample(waveform, orig_sr=source_sr, target_sr=REFERENCE_SAMPLE_RATE)
        reference_path = REFERENCE_DIR / sid / 'reference.wav'
        reference_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = reference_path.with_name('reference.partial.wav')
        sf.write(temporary, waveform, REFERENCE_SAMPLE_RATE, subtype='PCM_16')
        if not reference_duration_is_valid(float(sf.info(temporary).duration)):
            temporary.unlink(missing_ok=True)
            raise ValueError('Thời lượng WAV reference sau resample không đạt ngưỡng.')
        temporary.replace(reference_path)
        record = {key: row[key] for key in ['split', 'province_name', 'region', 'filename', 'speakerID', 'source_parquet']}
        record.update({'scenario_id': sid, 'gender': int(row['gender']),
                       'province_code': int(row['province_code']),
                       'row_group': int(row['row_group']), 'row_in_group': int(row['row_in_group']),
                       'dataset_id': VIMD_DATASET_ID, 'dataset_revision': VIMD_REVISION,
                       'ref_text_original': row['text'], 'ref_text_tts': row['text'].strip(),
                       'ref_audio_relative': reference_path.relative_to(OUTPUT_DIR).as_posix(),
                       'source_sample_rate': source_sr, 'sample_rate': REFERENCE_SAMPLE_RATE,
                       'audio_sha256': file_sha256(reference_path), 'metrics': metrics,
                       'request_fingerprint': fingerprint(reference_request(spec))})
        return record
    raise ValueError(f'{sid}: chưa tìm thấy reference nguyên bản hợp lệ trong {len(groups)} group. '
                     f'Mẫu bị loại: {rejected[:12]}. Chọn filename khác hoặc tăng MAX_REFERENCE_GROUPS.')

def prepare_references():
    bank = json.loads(REFERENCE_BANK_FILE.read_text(encoding='utf-8')) if REFERENCE_BANK_FILE.is_file() else {}
    reader = AudioGroupReader(HF_FS)
    errors = []
    for sid in tqdm(SELECTED_IDS, desc='Chọn giọng reference'):
        spec = VOICE_SPECS[sid]
        existing = bank.get(sid)
        cached_audio = OUTPUT_DIR / existing['ref_audio_relative'] if existing else None
        if (existing and existing.get('request_fingerprint') == fingerprint(reference_request(spec))
                and cached_audio.is_file() and file_sha256(cached_audio) == existing.get('audio_sha256')):
            continue
        try:
            bank[sid] = choose_reference(spec, reader)
            atomic_json(REFERENCE_BANK_FILE, bank)
            print(f"{sid}: {bank[sid]['filename']} / {bank[sid]['speakerID']} / {bank[sid]['metrics']['duration_seconds']:.2f}s")
        except Exception as exc:
            errors.append({'scenario_id': sid, 'error': f'{type(exc).__name__}: {exc}'})
            print(errors[-1])
    reader.table = None
    gc.collect()
    atomic_json(REFERENCE_DIR / 'reference_errors.json', errors)
    if errors:
        raise RuntimeError(f'{len(errors)} scenario chưa có reference đúng cấu hình. Xem references/reference_errors.json rồi chạy lại ô này.')
    return {sid: bank[sid] for sid in SELECTED_IDS}

REFERENCES = {}  # Xóa reference trong RAM của lần chạy trước nếu lần chọn mới bị lỗi.
REFERENCES = prepare_references()
reference_summary = pd.DataFrame([
    {'scenario_id': sid, 'gender': VOICE_SPECS[sid]['gender_label'],
     'province_requested': VOICE_SPECS[sid]['province_requested'],
     'region_requested': VOICE_SPECS[sid]['region_requested'],
     'province_vimd': ref['province_name'], 'region_vimd': ref['region'],
     'speakerID': ref['speakerID'], 'filename': ref['filename'],
     'duration_seconds': ref['metrics']['duration_seconds']}
    for sid, ref in REFERENCES.items()
])
reference_summary.to_csv(REFERENCE_DIR / 'reference_summary.csv', index=False, encoding='utf-8-sig')
display(reference_summary)


## 7. Nghe thử giọng mẫu và xem transcript

In [ ]:
for sid in PREVIEW_SCENARIO_IDS:
    if sid not in REFERENCES:
        continue
    ref = REFERENCES[sid]
    print(f"{sid} | {VOICE_SPECS[sid]['gender_label']} | {ref['province_name']} | {ref['speakerID']}")
    print(ref['ref_text_original'])
    display(Audio(filename=str(OUTPUT_DIR / ref['ref_audio_relative'])))

## 8. Nạp VoxCPM2 và cache giọng

Tải snapshot đúng `MODEL_REVISION`, gồm weights, tokenizer và `audiovae.pth`.
Tạo thư mục runtime dùng chung weights với snapshot; chỉ `config.json` runtime chọn dtype thích hợp GPU.
Snapshot gốc được giữ nguyên. API không có tham số dtype trực tiếp nên dùng cấu hình local.
Nạp `device='cuda:0'`, tắt denoiser và mặc định tắt compile/warm-up.

Cache giọng dùng hai đường dẫn tới cùng WAV + transcript ViMD.
Gọi `tts_model.generate_with_prompt_cache()` để tránh encode lại reference ở mỗi lượt.
Đây là API của `voxcpm==2.0.3` đã ghim; model trả về tuple tensor, output sample rate lấy từ model.
Đổi model/dtype sẽ giải phóng engine cũ trước khi nạp lại.


In [ ]:
def resolve_model_dtype():
    native_bf16 = (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
                   and torch.cuda.is_bf16_supported())
    if MODEL_DTYPE == 'auto':
        return 'bfloat16' if native_bf16 else 'float32'
    if MODEL_DTYPE == 'bfloat16' and not native_bf16:
        raise ValueError('GPU hiện tại không có native BF16. Dùng MODEL_DTYPE="auto" hoặc "float32".')
    return MODEL_DTYPE


def engine_configuration():
    return {'model_repo_id': MODEL_REPO_ID, 'model_revision': MODEL_REVISION,
            'voxcpm_version': VOXCPM_PACKAGE_VERSION, 'transformers_version': TRANSFORMERS_VERSION,
            'dtype': resolve_model_dtype(), 'optimize': OPTIMIZE_MODEL,
            'device': 'cuda:0', 'load_denoiser': False}


def prepare_runtime_model(snapshot_dir, runtime_dir, dtype):
    snapshot_dir, runtime_dir = Path(snapshot_dir), Path(runtime_dir)
    config = json.loads((snapshot_dir / 'config.json').read_text(encoding='utf-8'))
    if config.get('architecture') != 'voxcpm2':
        raise ValueError('Snapshot không có architecture=voxcpm2.')
    runtime_dir.mkdir(parents=True, exist_ok=True)
    names = ['model.safetensors', 'audiovae.pth', 'tokenizer.json',
             'tokenizer_config.json', 'special_tokens_map.json', 'tokenization_voxcpm2.py']
    for name in names:
        source = snapshot_dir / name
        target = runtime_dir / name
        if not source.is_file():
            raise FileNotFoundError(f'Snapshot thiếu {name}.')
        if not target.exists():
            try:
                target.symlink_to(source.resolve())
            except OSError:
                shutil.copy2(source, target)
    config['dtype'] = dtype
    config['device'] = 'cuda:0'
    atomic_json(runtime_dir / 'config.json', config)
    return runtime_dir


def get_voice_prompt(engine, ref_file, ref_text):
    ref_file = Path(ref_file)
    info = sf.info(ref_file)
    model = engine.tts_model
    if not reference_duration_is_valid(float(info.duration)):
        raise ValueError(f'Reference {info.duration:.3f}s ngoài khoảng '
                         f'[{REF_MIN_SECONDS}, {REF_MAX_SECONDS}]s. Chạy lại ô chọn reference.')
    ref_text = ref_text.strip()
    if not ref_text:
        raise ValueError('Transcript reference không được rỗng.')
    if info.channels != 1 or info.samplerate != model.audio_vae.sample_rate:
        raise ValueError('Reference cần mono / 16 kHz. Chạy lại ô chọn reference.')
    key = fingerprint({'engine': engine_configuration(), 'audio_sha256': file_sha256(ref_file),
                       'ref_text': ref_text, 'mode': 'ref_continuation', 'trim_silence_vad': False})
    cache = getattr(engine, '_vimd_prompt_cache', None)
    if cache is None:
        cache = engine._vimd_prompt_cache = {}
    if key in cache:
        return cache[key]
    with torch.inference_mode():
        prompt = model.build_prompt_cache(
            prompt_text=ref_text + ' ',  # Tách transcript mẫu khỏi câu mới khi model nối text.
            prompt_wav_path=str(ref_file), reference_wav_path=str(ref_file),
            trim_silence_vad=False,
        )
    if prompt.get('mode') != 'ref_continuation':
        raise ValueError('Prompt cache cần mode ref_continuation của VoxCPM2.')
    patch_seconds = model.patch_size * model.chunk_size / model.audio_vae.sample_rate
    ref_seconds = round(int(prompt['ref_audio_feat'].shape[0]) * patch_seconds, 9)
    continuation_seconds = round(int(prompt['audio_feat'].shape[0]) * patch_seconds, 9)
    if min(ref_seconds, continuation_seconds) + 1e-6 < REF_MIN_SECONDS:
        raise ValueError('Đặc trưng giọng sau encode không đạt ngưỡng thời lượng audio mẫu.')
    stats = {'original_duration_seconds': float(info.duration),
             'encoded_reference_seconds': ref_seconds, 'encoded_continuation_seconds': continuation_seconds,
             'mode': 'ref_continuation', 'trim_silence_vad': False, 'denoise': False}
    cache[key] = (prompt, stats)
    return cache[key]


def infer_with_full_reference(engine, *, ref_file, ref_text, gen_text, seed):
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    prompt, prompt_stats = get_voice_prompt(engine, ref_file, ref_text)
    with torch.inference_mode():
        result = engine.tts_model.generate_with_prompt_cache(
            target_text=re.sub(r'\s+', ' ', gen_text).strip(), prompt_cache=prompt,
            **GENERATION_OPTIONS,
        )
    if not isinstance(result, tuple) or len(result) != 3:
        raise ValueError('VoxCPM2 cần trả về tuple (waveform, text_tokens, audio_features).')
    audio, _, generated_features = result
    if not isinstance(audio, torch.Tensor):
        raise ValueError('Waveform VoxCPM2 cần là torch.Tensor.')
    if int(generated_features.shape[0]) >= GENERATION_OPTIONS['max_len']:
        raise ValueError('Generation chạm MAX_GEN_TOKENS; kiểm tra câu hoặc tăng giới hạn rồi chạy lại.')
    waveform = audio.detach().float().cpu()
    if waveform.ndim == 2 and waveform.shape[0] == 1:
        waveform = waveform.squeeze(0)
    return waveform.numpy(), int(engine.tts_model.sample_rate), dict(prompt_stats)


def load_engine():
    if not torch.cuda.is_available():
        raise RuntimeError('Chưa có GPU CUDA. Bật Accelerator → GPU trong Kaggle Settings / Session options.')
    from voxcpm import VoxCPM

    dtype = resolve_model_dtype()
    snapshot_dir = MODEL_CACHE_DIR / MODEL_REVISION / 'snapshot'
    snapshot_download(
        repo_id=MODEL_REPO_ID, revision=MODEL_REVISION, local_dir=snapshot_dir,
        allow_patterns=['*.json', '*.safetensors', 'audiovae.pth', 'tokenization_voxcpm2.py', 'README.md'],
        token=os.getenv('HF_TOKEN'),
    )
    runtime_dir = prepare_runtime_model(snapshot_dir, MODEL_CACHE_DIR / MODEL_REVISION / dtype, dtype)
    print('GPU:', torch.cuda.get_device_name(0))
    print('VoxCPM2 dtype:', dtype)
    print('Runtime model:', runtime_dir)
    engine = VoxCPM.from_pretrained(
        str(runtime_dir), load_denoiser=False, optimize=OPTIMIZE_MODEL,
        device='cuda:0', local_files_only=True,
    )
    if int(engine.tts_model.sample_rate) != 48000 or int(engine.tts_model.audio_vae.sample_rate) != REFERENCE_SAMPLE_RATE:
        raise ValueError('Cần VoxCPM2 nhận reference 16 kHz và xuất audio 48 kHz.')
    if engine.tts_model.config.dtype != dtype:
        raise ValueError('Dtype model không khớp cấu hình runtime.')
    return engine


if RUN_TTS:
    engine_config_key = fingerprint(engine_configuration())
    if globals().get('_ENGINE_CONFIG_KEY') != engine_config_key or globals().get('ENGINE') is None:
        globals().pop('ENGINE', None)
        gc.collect()
        torch.cuda.empty_cache()
        ENGINE = load_engine()
        _ENGINE_CONFIG_KEY = engine_config_key
    else:
        print('Dùng lại VoxCPM2 và prompt cache trong session.')
else:
    print('RUN_TTS=False: đã chuẩn bị dữ liệu/reference, chưa nạp checkpoint.')


## 9. Sinh một WAV cho mỗi lượt và lưu tiến độ

WAV PCM 16-bit mono / **48 kHz**, chỉ chứa câu cần sinh (model loại phần audio prompt khi decode).
Mỗi `customer_turns` là một lần generation và một file. Các câu hiện tại ngắn; không tự chia/ghép câu.
Seed ổn định theo `task_id`. Fingerprint gồm checkpoint/package/dtype, tham số sinh, câu đọc và SHA reference.
Đổi cấu hình/reference sẽ sinh lại WAV liên quan; output VoxCPM2 nằm trong thư mục riêng.

Lưu manifest sau từng lượt. Lượt lỗi được giữ để thử lại, các lượt khác tiếp tục.
VoxCPM2 tự retry bad case theo cấu hình. NaN/Inf/im lặng hoặc chạm giới hạn generation đều được đánh lỗi.
Chạy lại ô này để retry; nếu đổi tham số, chạy lại từ phần 2.
`reference_prompt` ghi thời lượng mẫu/đặc trưng, chế độ cloning và trạng thái VAD/denoise.
`MAX_TURNS=3` chỉ giới hạn lượt thử; các lượt chưa chạy vẫn được giữ với trạng thái pending.


In [ ]:
MANIFEST_FILE = OUTPUT_DIR / 'manifest.json'
INFERENCE_CONFIG = {
    **engine_configuration(), 'generation_options': dict(GENERATION_OPTIONS), 'base_seed': BASE_SEED,
    'cloning_mode': 'ref_continuation', 'reference_sample_rate': REFERENCE_SAMPLE_RATE,
    'reference_preprocessing': 'full_audio', 'trim_silence_vad': False,
    'normalize': False, 'denoise': False, 'pipeline_version': 1,
}

def new_result(task, ref):
    seed = (BASE_SEED + int(hashlib.sha256(task['task_id'].encode()).hexdigest()[:8], 16)) % (2 ** 31)
    signature = fingerprint({'task_id': task['task_id'], 'text_tts': task['text_tts'],
                             'reference_audio_sha256': ref['audio_sha256'],
                             'reference_text': ref['ref_text_tts'], 'config': INFERENCE_CONFIG, 'seed': seed})
    return {**task, 'voice_requested': VOICE_SPECS[task['scenario_id']],
            'reference': ref, 'seed': seed, 'input_fingerprint': signature, 'status': 'pending'}

def can_resume(row, expected, output_file):
    return (row.get('status') in {'generated', 'resumed'}
            and row.get('input_fingerprint') == expected['input_fingerprint']
            and output_file.is_file() and output_file.stat().st_size > 44
            and file_sha256(output_file) == row.get('output_sha256'))

def run_generation(engine):
    for sid in SELECTED_IDS:
        ref = REFERENCES.get(sid)
        if not ref or ref.get('request_fingerprint') != fingerprint(reference_request(VOICE_SPECS[sid])):
            raise RuntimeError(f'{sid}: chưa có reference đúng cấu hình hiện tại; chạy lại ô chọn reference.')
        if file_sha256(OUTPUT_DIR / ref['ref_audio_relative']) != ref['audio_sha256']:
            raise RuntimeError(f'{sid}: reference WAV đã đổi/hỏng; chạy lại ô chọn reference.')
    old_manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8')) if MANIFEST_FILE.is_file() else {}
    previous = old_manifest.get('results', {})
    results = {}
    for task in TASKS:
        row = new_result(task, REFERENCES[task['scenario_id']])
        prior = previous.get(task['task_id'], {})
        if can_resume(prior, row, OUTPUT_DIR / task['output_relative']):
            row.update({key: prior[key] for key in ['sample_rate', 'duration_seconds', 'bytes', 'output_sha256']})
            if 'reference_prompt' in prior:
                row['reference_prompt'] = prior['reference_prompt']
            row['status'] = 'resumed'
        results[task['task_id']] = row
    manifest = {'format_version': 1, 'scenario_ids': SELECTED_IDS, 'input_directory': str(DATASET_DIR),
                'output_directory': str(OUTPUT_DIR), 'expected_turns': len(TASKS),
                'inference_config': INFERENCE_CONFIG,
                'source_licenses': {'vimd': 'CC-BY-NC-ND-4.0', 'checkpoint': 'Apache-2.0', 'voxcpm_code': 'Apache-2.0'},
                'results': results}

    def persist():
        manifest['updated_at_utc'] = datetime.now(timezone.utc).isoformat()
        atomic_json(MANIFEST_FILE, manifest)

    persist()
    selected_tasks = TASKS if MAX_TURNS is None else TASKS[:MAX_TURNS]
    for task in tqdm(selected_tasks, desc='Sinh WAV từng customer_turn'):
        row = results[task['task_id']]
        output_file = OUTPUT_DIR / task['output_relative']
        if row['status'] == 'resumed' and not OVERWRITE:
            continue
        output_file.parent.mkdir(parents=True, exist_ok=True)
        temporary = output_file.with_name(output_file.stem + '.partial.wav')
        temporary.unlink(missing_ok=True)
        ref = row['reference']
        try:
            waveform, sample_rate, prompt_stats = infer_with_full_reference(
                engine, ref_file=str(OUTPUT_DIR / ref['ref_audio_relative']),
                ref_text=ref['ref_text_tts'], gen_text=row['text_tts'], seed=row['seed'],
            )
            waveform = np.asarray(waveform, dtype=np.float32)
            if waveform.ndim != 1 or len(waveform) == 0 or not np.isfinite(waveform).all():
                raise ValueError('Model trả về waveform rỗng/không mono/NaN/Inf.')
            if int(sample_rate) <= 0 or np.max(np.abs(waveform)) < 1e-6:
                raise ValueError('Model trả về sample rate không hợp lệ hoặc toàn im lặng.')
            sf.write(temporary, waveform, int(sample_rate), subtype='PCM_16')
            info = sf.info(temporary)
            if info.frames <= 0 or info.channels != 1:
                raise ValueError('WAV không hợp lệ sau khi ghi.')
            temporary.replace(output_file)
            row.update({'status': 'generated', 'reference_prompt': prompt_stats,
                        'sample_rate': int(info.samplerate),
                        'duration_seconds': float(info.duration), 'bytes': output_file.stat().st_size,
                        'output_sha256': file_sha256(output_file)})
            row.pop('error', None)
        except Exception as exc:
            row['status'] = 'error'
            row['error'] = f'{type(exc).__name__}: {exc}'
            print(f"Lỗi {task['task_id']}: {row['error']}")
        finally:
            temporary.unlink(missing_ok=True)
            persist()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return manifest

if RUN_TTS:
    MANIFEST = run_generation(ENGINE)
    status_counts = pd.Series([row['status'] for row in MANIFEST['results'].values()]).value_counts()
    display(status_counts.rename('turns').to_frame())
    unfinished = [key for key, row in MANIFEST['results'].items() if row['status'] not in {'generated', 'resumed'}]
    if unfinished:
        print(f'Còn {len(unfinished)} lượt lỗi/chưa sinh. Chạy lại sau khi sửa lỗi hoặc đặt MAX_TURNS=None.')
        print(unfinished[:20])
    else:
        print(f'Hoàn thành {len(TASKS)} WAV cho {len(SELECTED_IDS)} scenario.')
    print('Manifest:', MANIFEST_FILE)
else:
    print('Đặt RUN_TTS=True, chạy lại ô nạp model và ô generation để sinh WAV.')


## 10. Kiểm tra đủ file, xuất CSV/JSONL và nghe thử kết quả

Chỉ các file trong manifest của lần chọn scenario/call hiện tại được tính vào kết quả.
`manifest.csv` và `manifest.jsonl` đều có một dòng cho mỗi lượt, kể cả lượt lỗi/chưa sinh;
lọc `status` bằng `generated`/`resumed` khi đưa vào pipeline huấn luyện hoặc đánh giá.
Kiểm tra này xác nhận file và index; độ đúng lời, accent và chất lượng giọng cần nghe/ASR riêng.

In [ ]:
def export_and_check(manifest):
    rows = []
    issues = []
    for key, row in manifest['results'].items():
        ref = row['reference']
        flat = {name: row.get(name) for name in [
            'task_id', 'scenario_id', 'call_id', 'turn_index', 'turn_number',
            'text_original', 'text_tts', 'output_relative', 'status', 'seed',
            'sample_rate', 'duration_seconds', 'bytes', 'output_sha256', 'error']}
        flat.update({'gender_requested': row['voice_requested']['gender_label'],
                     'province_requested': row['voice_requested']['province_requested'],
                     'region_requested': row['voice_requested']['region_requested'],
                     'province_vimd': ref['province_name'], 'region_vimd': ref['region'],
                     'speakerID': ref['speakerID'], 'reference_filename': ref['filename'],
                     'reference_audio_relative': ref['ref_audio_relative'],
                     'reference_text': ref['ref_text_original'], 'reference_split': ref['split']})
        rows.append(flat)
        if row['status'] not in {'generated', 'resumed'}:
            continue
        path = OUTPUT_DIR / row['output_relative']
        try:
            info = sf.info(path)
            if info.frames <= 0 or info.channels != 1 or info.samplerate != row['sample_rate']:
                raise ValueError('WAV header/sample rate không khớp manifest.')
            if file_sha256(path) != row['output_sha256']:
                raise ValueError('Checksum WAV không khớp manifest.')
        except Exception as exc:
            issues.append({'task_id': key, 'error': str(exc)})
    frame = pd.DataFrame(rows)
    if frame.output_relative.duplicated().any() or frame.task_id.duplicated().any():
        raise ValueError('Tên file hoặc task_id bị trùng.')
    frame.to_csv(OUTPUT_DIR / 'manifest.csv', index=False, encoding='utf-8-sig')
    with (OUTPUT_DIR / 'manifest.jsonl').open('w', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
    atomic_json(OUTPUT_DIR / 'validation_issues.json', issues)
    if issues:
        raise RuntimeError(f'{len(issues)} WAV không khớp manifest; xem validation_issues.json và chạy lại generation.')
    return frame

if RUN_TTS:
    MANIFEST = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
    MANIFEST_TABLE = export_and_check(MANIFEST)
    display(MANIFEST_TABLE.groupby('scenario_id').agg(
        expected=('task_id', 'size'),
        completed=('status', lambda values: values.isin(['generated', 'resumed']).sum()),
        seconds=('duration_seconds', 'sum')))
    completed = MANIFEST_TABLE[MANIFEST_TABLE.status.isin(['generated', 'resumed'])]
    for row in completed.head(3).to_dict('records'):
        print(row['output_relative'])
        print('Câu gốc:', row['text_original'])
        print('Câu đọc:', row['text_tts'])
        display(Audio(filename=str(OUTPUT_DIR / row['output_relative'])))

## 11. Đóng gói kết quả và tải về

`RUN_MODE='full'` đặt `EXPORT_ZIP=True` tự động. ZIP chứa WAV thành công, reference,
metadata và cache metadata ViMD để chạy tiếp; không chứa checkpoint/source/cache model.
Nếu còn lượt lỗi/chưa sinh, dừng export để tránh xuất thiếu. Lưu phiên thành công bằng
**Save Version → Save & Run All**, rồi mở tab **Output** và tải `f5_vimd_customer_turns.zip`.
Phiên tương tác: cũng có thể tải các file hiện tại từ phần output/file browser của editor
trước khi dừng session. Không dựa vào việc working directory tự tồn tại ở phiên mới.

In [ ]:
if EXPORT_ZIP and RUN_TTS:
    import zipfile
    manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
    export_and_check(manifest)
    missing = [key for key, row in manifest['results'].items() if row['status'] not in {'generated', 'resumed'}]
    if missing:
        raise RuntimeError(f'Còn {len(missing)} lượt chưa hoàn thành; sinh đủ trước khi export ZIP.')
    paths = {row['output_relative'] for row in manifest['results'].values()}
    paths.update(row['reference']['ref_audio_relative'] for row in manifest['results'].values())
    paths.update(['manifest.json', 'manifest.csv', 'manifest.jsonl', 'validation_issues.json',
                  'references/reference_bank.json', 'references/reference_summary.csv'])
    paths.update(['cache/vimd_metadata.parquet', 'cache/vimd_metadata_config.json'])
    zip_path = OUTPUT_DIR / 'voxcpm2_vimd_customer_turns.zip'
    partial_zip = zip_path.with_name(zip_path.name + '.partial')
    with zipfile.ZipFile(partial_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for relative in sorted(paths):
            archive.write(OUTPUT_DIR / relative, arcname=relative)
    partial_zip.replace(zip_path)
    print('ZIP:', zip_path, f'({zip_path.stat().st_size / 1024 ** 2:.1f} MB)')
elif not RUN_TTS:
    print('RUN_TTS=False: chưa sinh WAV, chưa đóng ZIP. Đặt RUN_TTS=True rồi chạy lại từ phần 8.')
else:
    print('Kết quả trong working:', OUTPUT_DIR)
    print("Sau khi nghe thử, đổi RUN_MODE='full' và chạy lại từ phần 2 để sinh đủ và xuất ZIP.")
